Gaussian elimination with partial pivoting

In [1]:
import numpy as np

def gaussian_elimination(A, b):
    n = len(b)
    Ab = np.hstack([A.astype(float), b.reshape(-1, 1).astype(float)])

    for k in range(n):
        max_row = k + np.argmax(np.abs(Ab[k:, k]))
        Ab[[k, max_row]] = Ab[[max_row, k]]

        if abs(Ab[k, k]) < 1e-12:
            raise ValueError(f"Matrix is singular or nearly singular at pivot {k}")

        for i in range(k + 1, n):
            m = Ab[i, k] / Ab[k, k]
            Ab[i, k:] -= m * Ab[k, k:]

    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (Ab[i, -1] - Ab[i, i+1:n] @ x[i+1:n]) / Ab[i, i]

    return x

In [2]:
A = np.array([[2, 1, -1],
              [-3, -1, 2],
              [-2, 1, 2]], dtype=float)
b = np.array([8, -11, -3], dtype=float)

x = gaussian_elimination(A, b)
print("Solution:", x)
print("Residual (A@x - b):", A @ x - b)

Solution: [ 2.  3. -1.]
Residual (A@x - b): [0.0000000e+00 0.0000000e+00 8.8817842e-16]


In [12]:
A.shape

(3, 3)

In [19]:
np.hstack([A,b.reshape(-1,1)])

array([[  2.,   1.,  -1.,   8.],
       [ -3.,  -1.,   2., -11.],
       [ -2.,   1.,   2.,  -3.]])

In [9]:
b.shape

(3,)

In [18]:
b.reshape(-1,1).shape

(3, 1)

LU decomposition

In [20]:
def lu_decompose(A):
    n = A.shape[0]
    L = np.eye(n)
    U = A.astype(float).copy()
    P = np.eye(n)

    for k in range(n):
        max_row = k + np.argmax(np.abs(U[k:, k]))
        if max_row != k:
            U[[k, max_row]] = U[[max_row, k]]
            P[[k, max_row]] = P[[max_row, k]]
            if k > 0:
                L[[k, max_row], :k] = L[[max_row, k], :k]

        for i in range(k + 1, n):
            L[i, k] = U[i, k] / U[k, k]
            U[i, k:] -= L[i, k] * U[k, k:]

    return P, L, U

def lu_solve(P, L, U, b):
    n = len(b)
    Pb = P @ b.astype(float)

    y = np.zeros(n)
    for i in range(n):
        y[i] = Pb[i] - L[i, :i] @ y[:i]

    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (y[i] - U[i, i+1:] @ x[i+1:]) / U[i, i]

    return x

In [21]:
A = np.array([[2, 1, -1],
              [-3, -1, 2],
              [-2, 1, 2]], dtype=float)
b = np.array([8, -11, -3], dtype=float)

P, L, U = lu_decompose(A)
x = lu_solve(P, L, U, b)

print("P =\n", P)
print("L =\n", np.round(L, 4))
print("U =\n", np.round(U, 4))
print("Solution:", x)
print("Residual (A@x - b):", A @ x - b)
print("Check P@A == L@U:", np.allclose(P @ A, L @ U))

P =
 [[0. 1. 0.]
 [0. 0. 1.]
 [1. 0. 0.]]
L =
 [[ 1.      0.      0.    ]
 [ 0.6667  1.      0.    ]
 [-0.6667  0.2     1.    ]]
U =
 [[-3.     -1.      2.    ]
 [ 0.      1.6667  0.6667]
 [ 0.      0.      0.2   ]]
Solution: [ 2.  3. -1.]
Residual (A@x - b): [0.0000000e+00 0.0000000e+00 4.4408921e-16]
Check P@A == L@U: True


Cholesky decomposition